> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 03 · Your first Strands agent

**CampusMate v0 (a chatbot, built in 4 lines)**  ·  ⏱️ Time budget: **15 min theory + 30 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

"AI agent" sounds complicated, but the first one takes four lines of Python. In this lab you start with those four lines and add **one idea at a time**: a personality, a conversation it remembers, a choice of model, the cost of each answer, and finally a tool it can use. By the end you'll know every moving part of CampusMate's brain.

### What you'll do
- Create an agent in 4 lines
- Give it a personality with a system prompt
- See it remember a conversation
- Choose a model and limit answer length
- See how many tokens (and rupees) an answer costs
- Give it its first tool

## Settings

In [1]:
# ---- Step 0: make sure the workshop libraries are in THIS notebook's Python (macOS, Windows, Linux) ----
import sys, subprocess, importlib, importlib.util
if sys.version_info < (3, 10):
    raise SystemExit(f"This notebook runs Python {sys.version.split()[0]}. Choose the .venv kernel (Python 3.10+) and run again.")
if any(importlib.util.find_spec(m) is None for m in ("strands", "strands_tools", "bedrock_agentcore", "mcp_proxy_for_aws", "boto3", "dotenv")):
    print("Installing the workshop libraries into", sys.executable, "(1-2 minutes, only the first time)...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                               "strands-agents==1.64.0", "strands-agents-tools==0.9.0", "bedrock-agentcore==1.25.4",
                               "mcp-proxy-for-aws==1.8.0", "boto3==1.43.126", "botocore==1.43.126",
                               "flask==3.1.3", "python-dotenv==1.2.4", "requests==2.34.2"])
        importlib.invalidate_caches()
    except subprocess.CalledProcessError:
        raise SystemExit("Could not install here. Choose the .venv kernel (VS Code: Select Kernel -> .venv; "
                         "Jupyter: start it from a terminal after activating .venv), then run this cell again.")
print("Libraries ready in:", sys.executable)

# Settings for this lab
import sys, json, time, uuid, pathlib

REGION = "us-east-1"                                 # the AWS region we use in this workshop
MODEL_ID = "us.amazon.nova-2-lite-v1:0"              # Amazon Nova 2 Lite (CampusMate's brain)
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"        # same model for quick experiments

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"                             # the fictional campus data
print("Region:", REGION, "| Model:", MODEL_ID)

Region: us-east-1 | Model: us.amazon.nova-2-lite-v1:0


## What is Strands?

**Strands Agents** is an open-source Python library from AWS for building AI agents. An agent is three things:

- a **model** (the brain): here, Amazon Nova 2 Lite running on **Amazon Bedrock**, AWS's service that hosts AI models (you pay per token)
- **instructions** (the system prompt): who the agent is and how it should behave
- **tools**: Python functions it may call to look things up or take actions

🏫 **Analogy:** Strands is like the college's online admission portal. You fill in one simple form; the portal deals with all the offices behind it.

## Step 1: an agent in 3 lines

In [2]:
from strands import Agent

agent = Agent(model="us.amazon.nova-2-lite-v1:0")
answer = agent("Hi! In two sentences, what can an AI assistant do for a college student?")

An AI assistant can help a college student by organizing study schedules, managing deadlines, and providing resources for research, thereby improving time management and academic performance. Additionally, it can offer personalized support for writing, problem-solving, and accessing course materials, enhancing the overall learning experience.

What just happened, line by line:

1. `from strands import Agent`: load the library.
2. `Agent(model=...)`: create an agent that uses Amazon Nova 2 Lite (a fast, low-cost model) on Amazon Bedrock.
3. `agent("...")`: send a message. The answer prints **as it's being written** (streaming), and is also saved in `answer`.

That's a working chatbot. Everything after this adds one idea at a time.

## Step 2: give it a personality

The **system prompt** is a standing instruction the agent follows in every reply, like a job description.

In [3]:
campusmate = Agent(
    model=CHEAP_MODEL_ID,
    system_prompt="You are CampusMate, a friendly assistant for AIDD students. Answer in 2-3 short sentences.",
)
answer = campusmate("What should I focus on in my first week of 3rd semester?")

Welcome to semester 3! Focus on reviewing last semester’s notes to refresh your memory and familiarize yourself with new course materials. Attend all classes and introduce yourself to professors to get off on the right foot.

## Step 3: it remembers the conversation

Keep talking to the **same** agent.

In [4]:
answer = campusmate("Hi, I'm Kiran and I love cloud computing.")
print("\n---")
answer = campusmate("What's my name, and which topic do I like?")
print("\n\nMessages the agent is keeping:", len(campusmate.messages))

Hi Kiran! Great to meet you. Cloud computing is an exciting field—perfect for AIDD students! Keep exploring and asking questions in class to deepen your understanding.
---
You're Kiran, and you love cloud computing. It's a fantastic area to focus on, especially with the growing opportunities in AI and data-driven cloud solutions!

Messages the agent is keeping: 6


Remember notebook 01: the model itself is **stateless**. The agent keeps the conversation in a list (`campusmate.messages`) and sends
the whole list every time. Create a new `Agent(...)` and you start a fresh conversation. Remembering things *across* conversations
(next week, next semester) needs real memory, which you'll add in Lab 6.

## Step 4: choose the model and the answer length

This workshop uses one model, **Amazon Nova 2 Lite** (`MODEL_ID`, also `CHEAP_MODEL_ID`): fast, low-cost and good with tools.
Bedrock hosts many others; you change models by changing one ID.

`BedrockModel` lets you choose the model **and** cap the answer length with `max_tokens`, which also caps the cost.
If an answer hits the cap, Strands stops with a `MaxTokensReachedException`: ask for a shorter answer, or raise the cap.

In [5]:
from strands.models import BedrockModel

smart_model = BedrockModel(model_id=MODEL_ID, max_tokens=1000)      # the smarter model, at most ~1000 tokens per answer
smart = Agent(model=smart_model, system_prompt="You are CampusMate. Be brief.")
answer = smart("Explain AI Agent to a first-year student using a hostel analogy, in 4 short sentences.")

Imagine an AI agent like a super-organized hostel manager: it handles tasks automatically, like assigning rooms based on preferences. It learns from residents’ habits to improve service over time. Just like a manager anticipates needs, an AI agent predicts what you might want next. It’s always on, working quietly to make your digital life easier.

## Step 5: what did that cost?

Every answer reports how many **tokens** went in (your message, the system prompt and the conversation so far) and came out.
`callback_handler=None` turns off the live printing, so we print the answer ourselves.

In [6]:
quiet = Agent(model=BedrockModel(model_id=CHEAP_MODEL_ID, streaming=False), callback_handler=None)   # streaming=False: token counts always come back
result = quiet("Give me three tips to keep my attendance above 75%.")
print(result)

usage = result.metrics.accumulated_usage
print("tokens in:", usage["inputTokens"], "| tokens out:", usage["outputTokens"])
cost = usage["inputTokens"] * 0.33 / 1_000_000 + usage["outputTokens"] * 2.75 / 1_000_000   # Nova 2 Lite prices (Nov 2026)
print(f"cost: ${cost:.6f}  (about Rs. {cost * 84:.3f})")

### **Three Tips to Keep Your Attendance Above 75%**

Maintaining consistent attendance—whether for school, work, or any committed program—requires planning, motivation, and a proactive approach. Here are **three effective tips** to help you keep your attendance above 75%:

---

#### **1. Create a Structured Schedule and Set Reminders**
  
**Why it works:** Consistency is key to good attendance. By planning ahead, you reduce the likelihood of missing sessions due to poor time management or forgetfulness.

**How to implement it:**
- **Plan your week:** Use a digital calendar (like Google Calendar) or a planner to block out every class, meeting, or work shift.
- **Set reminders:** Schedule notifications 1–2 days before and the morning of each session.
- **Prioritize key commitments:** Treat important sessions (e.g., exams, presentations, team meetings) as non-negotiable appointments.

**Bonus tip:** Place your schedule where you’ll see it daily—on your phone lock screen, fridge, or bathr

## Step 6: its first tool

A **tool** is an ordinary Python function with `@tool` on top. The agent reads the function's name and docstring, decides by itself
when it needs it, and Strands runs it. Watch the agent call `get_credits` twice and add the results.

In [7]:
from strands import tool

@tool
def get_credits(course_code: str) -> int:
    """Return the number of credits for an AIDD course code such as AIDD303."""
    credits = {"AIDD201": 3, "AIDD202": 2, "AIDD203": 4, "AIDD303": 4, "AIDD305": 3}
    return credits.get(course_code.upper(), 0)

helper = Agent(model=CHEAP_MODEL_ID, tools=[get_credits])
answer = helper("How many credits are AIDD303 and AIDD202 together?")
print("\n\nTools used:", list(helper.event_loop_metrics.tool_metrics))


Tool #1: get_credits

Tool #2: get_credits
AIDD303 is worth **4 credits** and AIDD202 is worth **2 credits**, so together they are worth **6 credits**.

Tools used: ['get_credits']


How the tool call works:

1. Strands tells the model "you have a tool called `get_credits`" (using the name and docstring).
2. The model replies "please run `get_credits('AIDD303')`" instead of answering.
3. Strands runs your Python function and hands the result back.
4. The model uses the result to write the final answer.

This **think → use a tool → look at the result** loop is what makes it an *agent* rather than a chatbot.
In notebook 04 you'll build real CampusMate tools that read the course list and the campus rules.

> Good to know: Amazon Bedrock also has **Guardrails** (filters for harmful content and personal data) and **Knowledge Bases**
> (search over your documents). We don't use them today, but they plug into the same kind of agent.

## ✍️ Your turn

Add a second tool `get_semester(course_code)` that returns the semester (AIDD2xx → 2, AIDD3xx → 3) and ask a question that needs both tools.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
@tool
def get_semester(course_code: str) -> int:
    """TODO: describe the tool for the model (what it returns, when to use it)."""
    return 0   # TODO: return 2 for AIDD2xx codes and 3 for AIDD3xx codes

MY_QUESTION = "TODO"   # TODO: e.g. "Which semester is AIDD303 in, and how many credits is it?"
if MY_QUESTION != "TODO":
    both = Agent(model=CHEAP_MODEL_ID, tools=[get_credits, get_semester])
    answer = both(MY_QUESTION)
else:
    print("Skipped: fill in get_semester and MY_QUESTION first.")

## ✅ Checkpoint

You are on track if:
- Step 1 printed a short answer
- Step 3: the agent remembered Kiran and cloud computing
- Step 5 printed token counts and a cost well under one rupee
- Step 6 listed `get_credits` under tools used, and the answer was 6 credits

**Stuck or behind?** Copy from `scripts/03_first_agent.py` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'strands_agents'` | Old tutorials use the wrong import | Use `from strands import Agent` |
| `AccessDeniedException ... bedrock:InvokeModel` | Your IAM user lacks Bedrock permissions | Your IAM user needs the Labs 1-4 permissions listed in SETUP.md |
| `AccessDeniedException ... model access` | Nova 2 Lite isn't enabled in your account | Bedrock console → **Model access** → enable Amazon Nova 2 Lite |
| `MaxTokensReachedException` | `max_tokens` too small (the model may think before answering) | Raise `max_tokens` (e.g. 1000) |
| The answer prints twice | The agent call is the last line of the cell, so Jupyter also shows its return value | Assign it: `answer = agent(...)` |